In [ ]:
!pip install --quiet pdfplumber pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 117.3 MB/s eta 0:00:00


Setting up paths clearly

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, re, json
from tqdm.auto import tqdm
import pdfplumber
import pandas as pd

BASE = "/content/drive/MyDrive/Semantic_search2"
DATA_DIR = os.path.join(BASE, "data")
OUT_DIR = os.path.join(BASE, "outputs")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

CONCORD_PDF = os.path.join(DATA_DIR, "NCO_2015_ConcordanceTable.pdf")
JOBDESC_PDF  = os.path.join(DATA_DIR, "NCO_2015_JobDesc.pdf")

CONCORD_CSV = os.path.join(DATA_DIR, "nco_concordance_cleaned.csv")
CONCORD_DESC_CSV = os.path.join(DATA_DIR, "nco_final.csv")

print("DATA_DIR:", DATA_DIR)
print("Expecting PDFs at:", CONCORD_PDF, "and", JOBDESC_PDF)
print("Outputs (csv):", CONCORD_CSV, CONCORD_DESC_CSV)

Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/Semantic_search2/data
Expecting PDFs at: /content/drive/MyDrive/Semantic_search2/data/NCO_2015_ConcordanceTable.pdf and /content/drive/MyDrive/Semantic_search2/data/NCO_2015_JobDesc.pdf
Outputs (csv): /content/drive/MyDrive/Semantic_search2/data/nco_concordance_cleaned.csv /content/drive/MyDrive/Semantic_search2/data/nco_final.csv


In [4]:
if not os.path.exists(CONCORD_PDF) or not os.path.exists(JOBDESC_PDF):
    raise FileNotFoundError("Place 'NCO_2015_ConcordanceTable.pdf' and 'NCO_2015_JobDesc.pdf' inside:\n" + DATA_DIR)

**CONCORDANCE TABLE**

In [ ]:
if os.path.exists(CONCORD_CSV):
    print("Concordance CSV already exists. Skipping concordance extraction.")
else:
    print("Extracting concordance table from PDF...")
    HEADER_PATTERN = r"NATIONAL CLASSIFICATION OF OCCUPATIONS\s*-\s*2015"
    FOOTER_PATTERN = r"VOLUME\s*I\s*\d+"
    FIRST_ROW_PATTERN = r"\bNCO\s*2015\b.*\bNCO\s*2004\b"
    def clean_text(text):
        text = re.sub(HEADER_PATTERN, "", text, flags=re.IGNORECASE)
        text = re.sub(FOOTER_PATTERN, "", text, flags=re.IGNORECASE)
        text = re.sub(FIRST_ROW_PATTERN, "", text, flags=re.IGNORECASE)
        return text.strip()

    all_rows = []
    with pdfplumber.open(CONCORD_PDF) as pdf:
        for page in tqdm(pdf.pages, desc="Processing concordance pages"):
            table = page.extract_table()
            if not table:
                tables = page.extract_tables() or []
                table = tables[0] if tables else None
            if not table:
                continue
            for row in table:
                if not any(row):
                    continue
                row = [clean_text(str(cell)) if cell else "" for cell in row]
                joint = " ".join(row)
                if "NCO 2015" in joint and "NCO 2004" in joint:
                    continue
                all_rows.append(row)

    # fix first-column empties
    for r in all_rows:
        if len(r) >= 1 and r[0].strip() == "":
            r[0] = "Job title"

    # ensure 4 columns per row
    cleaned = []
    for r in all_rows:
        if len(r) < 4:
            r = (r + [""]*4)[:4]
        else:
            r = r[:4]
        cleaned.append(r)

    df_conc = pd.DataFrame(cleaned, columns=["Category", "NCO 2015 Code", "Title", "NCO 2004 Code"])
    df_conc.to_csv(CONCORD_CSV, index=False)
    print("Saved concordance CSV ->", CONCORD_CSV)

Extracting concordance table from PDF...


Processing concordance pages:   0%|          | 0/201 [00:00<?, ?it/s]

Saved concordance CSV -> /content/drive/MyDrive/Semantic_search2/data/nco_concordance_cleaned.csv


**JOB DESCRIPTIONS**

In [ ]:
if os.path.exists(CONCORD_DESC_CSV):
    print("Concordance+desc CSV already exists. Skipping jobdesc extraction and mapping.")
else:
    print("Extracting job descriptions from JOBDESC PDF and mapping to concordance...")

    def remove_headers_footers(text):
        patterns = [
            r"NATIONAL CLASSIFICATIO",
            r"VO\s*OF OCCUPATIONS - 2015",
            r"\bN\s+OF\s+OCCUPATIONS\.?",
            r"OF OCCUPATIONS - 2015",
            r"UME III\s*\d+",
        ]
        t = text
        for p in patterns:
            t = re.sub(p, "", t, flags=re.IGNORECASE)
        return t.strip()

    def extract_two_columns(page):
        width = page.width
        mid = width / 2
        left_box = (0, 0, mid, page.height)
        right_box = (mid, 0, width, page.height)
        left_text = page.within_bbox(left_box).extract_text() or ""
        right_text = page.within_bbox(right_box).extract_text() or ""
        if not left_text and not right_text:
            return page.extract_text() or ""
        return (left_text.strip() + "\n\n" + right_text.strip()).strip()

    # extract all pages
    pages_text = []
    with pdfplumber.open(JOBDESC_PDF) as pdf:
        for page in tqdm(pdf.pages, desc="Extracting JobDesc pages"):
            t = extract_two_columns(page)
            pages_text.append(remove_headers_footers(t))

    # join pages and add newlines before code markers
    clean_pages = [re.sub(r"\s*\n\s*", " ", p).strip() for p in pages_text]
    joined = " ".join(clean_pages)
    patterns_before = [
        r"(Division\s+\d\b)",
        r"(Sub Division\s+\d{2}\b)",
        r"(Group\s+\d{3}\b)",
        r"(Family\s+\d{4}\b)",
        r"(\d{4}\.\d{4}\b)"
    ]
    for p in patterns_before:
        joined = re.sub(p, r"\n\1", joined)
    joined = re.sub(r"^\n+", "", joined)

    # parse blocks using regex with named groups
    PATTERNS = {
        "division": re.compile(r"(?P<header>Division\s+(?P<code>\d))\s+(?P<desc>.*?)(?=\n(Sub Division|Division|\Z))", re.S | re.I),
        "subdivision": re.compile(r"(?P<header>Sub Division\s+(?P<code>\d{2}))\s+(?P<desc>.*?)(?=\n(Group|Sub Division|Division|\Z))", re.S | re.I),
        "group": re.compile(r"(?P<header>Group\s+(?P<code>\d{3}))\s+(?P<desc>.*?)(?=\n(Family|Group|Division|\Z))", re.S | re.I),
        "family": re.compile(r"(?P<header>Family\s+(?P<code>\d{4}))\s+(?P<desc>.*?)(?=\n(\d{4}\.|Family|Group|\Z))", re.S | re.I),
        "job": re.compile(r"(?P<header>(?P<code>\d{4}\.\d{4}))\s+(?P<desc>.*?)(?=\n(\d{4}\.|Family|Group|Division|\Z))", re.S)
    }

    def normalize_job_code(code):
        code = code.strip()
        if "." not in code:
            return code
        left, right = code.split(".", 1)
        right = right.rstrip("0")
        if len(right) < 2:
            right = right.ljust(2, "0")
        return f"{left}.{right}"

    extracted = []
    for ptype, patt in PATTERNS.items():
        for m in patt.finditer(joined):
            g = m.groupdict()
            code = (g.get("code") or "").strip()
            desc = (g.get("desc") or "").strip()
            if ptype == "job" and code:
                code = normalize_job_code(code)
            extracted.append({"type": ptype, "code": code, "description": desc})

    lookup = {}
    for item in extracted:
        c = item["code"]
        if not c:
            continue
        # prefer longer desc if duplicates
        prev = lookup.get(c, "")
        if len(item["description"]) > len(prev):
            lookup[c] = item["description"]

    # Map descriptions to concordance dataframe
    df_conc = pd.read_csv(CONCORD_CSV)
    df_conc["Description"] = ""
    for i, row in df_conc.iterrows():
        code = str(row.get("NCO 2015 Code", "")).strip()
        candidates = [code, code.replace(" ", ""), code.replace("-", "")]
        if "." in code:
            try_norm = normalize_job_code(code)
            candidates.insert(0, try_norm)
        plain = re.sub(r"[^\d]", "", code)
        if len(plain) >= 6:
            candidates.append(plain[:6])
        chosen = ""
        for cand in candidates:
            if cand in lookup:
                chosen = lookup[cand]; break
        if chosen:
            df_conc.at[i, "Description"] = chosen

    df_conc.to_csv(CONCORD_DESC_CSV, index=False)
    print("Saved concordance+desc CSV ->", CONCORD_DESC_CSV)

Extracting job descriptions from JOBDESC PDF and mapping to concordance...


Extracting JobDesc pages:   0%|          | 0/1152 [00:00<?, ?it/s]

Saved concordance+desc CSV -> /content/drive/MyDrive/Semantic_search2/data/nco_concordance_desc.csv


In [ ]:
import pandas as pd

# Load the CSV generated earlier
df = pd.read_csv("/content/drive/MyDrive/Semantic_search2/data/nco_concordance_desc.csv")

# Detect rows where the first column contains the literal text "Job title"
# (Assuming the first column name is something like "Category" or similar)
first_col = df.columns[0]

# Keep only rows where the first column value equals "Job title"
df_job = df[df[first_col].astype(str).str.strip() == "Job title"]

# Save filtered CSV
output_path = "/content/drive/MyDrive/Semantic_search2/data/nco_final.csv"
df_job.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: /content/drive/MyDrive/Semantic_search2/data/nco_final.csv


**SEMANTIC SEARCH**

In [5]:
# Part 2 — SBERT encoding, optional fine-tune, build FAISS (saves to outputs)
!pip install --upgrade pip -q
!pip install sentence-transformers faiss-cpu pandas tqdm rapidfuzz -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.5 MB/s eta 0:00:00


In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
import os, json, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

BASE = "/content/drive/MyDrive/Semantic_search2"
DATA_DIR = os.path.join(BASE, "data")
OUT_DIR = os.path.join(BASE, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

JOB_CSV = os.path.join(DATA_DIR, "nco_final.csv")
INDEX_DF_OUT = os.path.join(OUT_DIR, "index_df_canonical.csv")
EMBED_PATH = os.path.join(OUT_DIR, "nco_embeddings.npy")
FAISS_PATH = os.path.join(OUT_DIR, "nco_faiss.index")
CODES_PATH = os.path.join(OUT_DIR, "nco_codes.json")
TITLES_PATH = os.path.join(OUT_DIR, "nco_titles.json")

assert os.path.exists(JOB_CSV), f"Run Part 1 first. Missing: {JOB_CSV}"

In [ ]:
# load and prepare index_df
df = pd.read_csv(JOB_CSV, dtype=str).fillna("")
def normalize_code(c):
    if not c: return ""
    return __import__('re').sub(r'[\s\.\-]', '', str(c))
df['NCO_Code'] = df['NCO 2015 Code'].apply(normalize_code)
def make_canonical(row):
    t = str(row['Title']).strip()
    d = str(row['Description']).strip()
    return (t + ". " + d) if (t and d) else (t or d)
df['canonical_text'] = df.apply(make_canonical, axis=1)
index_df = df[['NCO_Code','Title','Description','canonical_text','Category']].copy().reset_index(drop=True)

# Save index_df snapshot (if not exists)
if not os.path.exists(INDEX_DF_OUT):
    index_df.to_csv(INDEX_DF_OUT, index=False)
    print("Saved index_df ->", INDEX_DF_OUT)
else:
    print("index_df already exists ->", INDEX_DF_OUT)

# If embeddings & faiss exist, skip heavy work
if os.path.exists(EMBED_PATH) and os.path.exists(FAISS_PATH):
    print("Embeddings and FAISS index already exist. Skipping encoding & index build.")
else:
    MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"
    print("Loading SBERT model:", MODEL_NAME)
    model = SentenceTransformer(MODEL_NAME)
    model.max_seq_length = 256

Saved index_df -> /content/drive/MyDrive/Semantic_search2/outputs/index_df_canonical.csv
Loading SBERT model: paraphrase-multilingual-mpnet-base-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Put this at top of the notebook (before model.fit)
import os
# disable wandb completely
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
# also ensure HF won't try to use wandb
os.environ["HF_WANDB_DISABLED"] = "true"

# (Then re-run your fine-tuning cell)

In [ ]:
# --------------------------------------------------------
# SAFE FINETUNE CELL — runs ONLY if finetuned model folder
# does NOT already exist.
# --------------------------------------------------------
from sentence_transformers import SentenceTransformer
import os

FINETUNE_DIR = os.path.join(OUT_DIR, "sbert_nco_finetuned")

# If folder already exists → load it and skip finetuning
if os.path.exists(FINETUNE_DIR):
    print("Fine-tuned model already exists. Loading it...")
    model = SentenceTransformer(FINETUNE_DIR)
    print("Loaded fine-tuned model from:", FINETUNE_DIR)

else:
    print("Fine-tuned model not found. Running finetuning now...")
    DO_FINETUNE = True

    if DO_FINETUNE:
        from sentence_transformers import InputExample, losses
        from torch.utils.data import DataLoader
        import random

        samples = []
        for _, r in index_df.iterrows():
            title = (r['Title'] or "")[:200]
            canon = (r['canonical_text'] or "")[:800]
            if title.strip() and canon.strip():
                samples.append(InputExample(texts=[title, canon]))

        if samples:
            random.shuffle(samples)
            train_loader = DataLoader(samples, shuffle=True, batch_size=16)
            train_loss = losses.MultipleNegativesRankingLoss(model)

            model.fit(
                train_objectives=[(train_loader, train_loss)],
                epochs=1,
                warmup_steps=10,
                show_progress_bar=True,
                output_path=FINETUNE_DIR
            )

            print("Fine-tuned model saved at:", FINETUNE_DIR)
            model = SentenceTransformer(FINETUNE_DIR)
    else:
        print("Skipping fine-tuning (DO_FINETUNE = False).")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Fine-tuned model not found. Running finetuning now...


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Fine-tuned model saved at: /content/drive/MyDrive/Semantic_search2/outputs/sbert_nco_finetuned


In [ ]:
texts = index_df['canonical_text'].fillna("").tolist()
print("Encoding", len(texts), "texts...")
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
print("Embeddings shape:", embeddings.shape)


Encoding 3460 texts...


Batches:   0%|          | 0/109 [00:00<?, ?it/s]

Embeddings shape: (3460, 768)


In [ ]:
# save embeddings & mappings
np.save(EMBED_PATH, embeddings)
with open(CODES_PATH, "w", encoding="utf8") as f: json.dump(index_df['NCO_Code'].fillna("").tolist(), f, ensure_ascii=False)
with open(TITLES_PATH, "w", encoding="utf8") as f: json.dump(index_df['Title'].fillna("").tolist(), f, ensure_ascii=False)
print("Saved embeddings & mappings to", OUT_DIR)

Saved embeddings & mappings to /content/drive/MyDrive/Semantic_search2/outputs


In [ ]:
# Build FAISS index
import faiss
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)
faiss.write_index(index, FAISS_PATH)
print("Built & saved FAISS index ->", FAISS_PATH)


Built & saved FAISS index -> /content/drive/MyDrive/Semantic_search2/outputs/nco_faiss.index


**INDICBERT- MULTILINGUAL INPUT**

In [ ]:
!pip install --quiet transformers torch scikit-learn joblib

In [ ]:
import os, json, time
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

BASE = "/content/drive/MyDrive/Semantic_search2"
OUT_DIR = os.path.join(BASE, "outputs")
INDEX_DF_PATH = os.path.join(OUT_DIR, "index_df_canonical.csv")
SBERT_EMB_PATH = os.path.join(OUT_DIR, "nco_embeddings.npy")
MAPPER_PATH = os.path.join(OUT_DIR, "indicbert_to_sbert_mapper.joblib")
META_PATH = os.path.join(OUT_DIR, "indicbert_mapper_meta.json")

INDICBERT_NAME = "ai4bharat/IndicBERTv2-MLM-only"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Tune if OOM
BATCH_SIZE = 64
TRAIN_SAMPLE = None   # set int like 5000 to speed up training
ALPHA = 1.0
VALIDATION_SPLIT = 0.05
MAX_LEN = 256

print("Part 3 starting. Device:", DEVICE)
assert os.path.exists(INDEX_DF_PATH), "index_df_canonical.csv missing (run Part 2 first)"
assert os.path.exists(SBERT_EMB_PATH), "nco_embeddings.npy missing (run Part 2 first)"

index_df = pd.read_csv(INDEX_DF_PATH, dtype=str).fillna("")
sbert_embs = np.load(SBERT_EMB_PATH)
print("Loaded index_df rows:", len(index_df), "SBERT emb shape:", sbert_embs.shape)

Part 3 starting. Device: cuda
Loaded index_df rows: 3460 SBERT emb shape: (3460, 768)


# **LOADING MODELS AND TESTING (SBERT)**

In [ ]:
# ----------------------------------------
# TEST FINE-TUNED SBERT AGAINST FAISS
# ----------------------------------------
import os, faiss, pandas as pd
from sentence_transformers import SentenceTransformer

OUT_DIR = "/content/drive/MyDrive/Semantic_search2/outputs"
FINETUNE_DIR = os.path.join(OUT_DIR, "sbert_nco_finetuned")
FAISS_PATH = os.path.join(OUT_DIR, "nco_faiss.index")
INDEX_DF_PATH = os.path.join(OUT_DIR, "index_df_canonical.csv")

# 1. Load fine-tuned model
assert os.path.exists(FINETUNE_DIR), "Fine-tuned model not found!"
model = SentenceTransformer(FINETUNE_DIR)
model.max_seq_length = 256
print("Loaded fine-tuned SBERT:", FINETUNE_DIR)

# 2. Load FAISS index + dataframe
index = faiss.read_index(FAISS_PATH)
index_df = pd.read_csv(INDEX_DF_PATH, dtype=str).fillna("")
print("Loaded FAISS + index_df. Vectors:", index.ntotal)

# Helper to search
def search_finetuned(query, k=5):
    q_emb = model.encode([query], normalize_embeddings=True)
    D, I = index.search(q_emb, k)
    rows = []
    for score, idx in zip(D[0], I[0]):
        rec = index_df.iloc[idx]
        rows.append({
            "score": round(float(score), 4),
            "NCO_Code": rec["NCO_Code"],
            "Title": rec["Title"],
            "Description": rec["Description"][:200]
        })
    return pd.DataFrame(rows)

# ----------------------------------------
# TEST QUERIES (modify freely)
# ----------------------------------------
queries = [
    "sewing machine operator",
    "truck driver",
    "security guard",
    "software engineer backend",
    "data entry operator",
    "electrician",
    "chef",
    "bank clerk"
]

for q in queries:
    print(f"\n Query: {q}")
    display(search_finetuned(q))


Loaded fine-tuned SBERT: /content/drive/MyDrive/Semantic_search2/outputs/sbert_nco_finetuned
Loaded FAISS + index_df. Vectors: 3460

 Query: sewing machine operator


,score,NCO_Code,Title,Description
0,0.7305,81530101,"Sewing Machine Operator, General","Sewing Machine Operator, General Sewing Machin..."
1,0.7283,81530103,Specialized Sewing Machine Operator,Specialized Sewing Machine Operator Specialize...
2,0.6517,815306,Embroidery Machine Operator (Semi-\nAutomatic),Embroidery-Machine Operator (Semi-Automatic) E...
3,0.6379,,Machine Operators,
4,0.5780,723318,"Mechanic, Sewing Machine","Mechanic, Sewing Machine Mechanic Sewing Machi..."



 Query: truck driver


,score,NCO_Code,Title,Description
0,0.8540,833201,Driver Truck,"Driver Truck Driver, Truck drives motor truck ..."
1,0.5974,833299,"Heavy Truck and Lorry Drivers, Other","Heavy Truck and Lorry Drivers, Other Heavy Tru..."
2,0.5047,834401,Lift Truck Operator,Lift Truck Operator Lift Truck Operator operat...
3,0.4708,834499,"Lift Truck Operator, Other","Lift Truck Operator, Other Lifting Truck Opera..."
4,0.4489,834213,Scoop Truck Operator,Scoop Truck Operator Scoop Truck Operator oper...



 Query: security guard


...

,score,NCO_Code,Title,Description
0,0.6694,741101,"Electrician, General","Electrician, General Electrician, General inst..."
1,0.6429,82120402,Electrical Assembly Operator,Electrical Assembly Operator Electrical Assemb...
2,0.5982,82122401,Electrical Technician,Electrical Technician Electrical Technician is...
3,0.5974,741102,Electrician (Mines),Electrician (Mines) Electrician (Mines) studie...
4,0.5821,74120701,"Electrician, Automobile","Electrician, Automobile Electrician, Automobil..."



 Query: chef


,score,NCO_Code,Title,Description
0,0.6065,343401,Head Cook/Head Chef,Head Cook/Head Chef Head Cook plans meals and ...
1,0.5404,511101,"Chief Steward, Ship/Chief Cook","Chief Steward, Ship/Chief Cook Chief Steward, ..."
2,0.4718,112031,Executive Chef,Executive Chef Executive Chef co-ordinates act...
3,0.3770,312236,"Supervisor and Foreman, Brush Making","Supervisor and Foreman, Brush Making Superviso..."
4,0.3674,312269,"Supervisor and Foreman, Food\nProcessing/Produ...","Supervisor and Foreman, Food Processing/Produc..."



 Query: bank clerk


,score,NCO_Code,Title,Description
0,0.7272,431102,Bank Clerk,Bank Clerk Bank Clerk maintains various accoun...
1,0.5322,421199,"Bank Tellers and Related Clerks, Other","Bank Tellers and Related Clerks, Other Tellers..."
2,0.5145,411001,"Clerk, General","Clerk, General Clerk, General performs variety..."
3,0.4878,441101,Library Clerk,Library Clerk Library Clerk; Library Assistant...
4,0.4843,431106,"Clerk, Cost Accounting","Clerk, Cost Accounting Clerk, Cost Accounting ..."


In [ ]:
import shutil, os
OUT = "/content/drive/MyDrive/Semantic_search2/outputs"
old = os.path.join(OUT, "indicbert_to_sbert_mapper.joblib")
bak = os.path.join(OUT, "indicbert_to_sbert_mapper.joblib.bak")
if os.path.exists(old) and not os.path.exists(bak):
    shutil.copy(old, bak); print("Backed up existing mapper ->", bak)
else:
    print("No existing mapper to backup or backup already exists.")


No existing mapper to backup or backup already exists.


In [ ]:
# Train MLP mapper (run now)
!pip install --quiet transformers torch scikit-learn joblib

import os, time, joblib, numpy as np, pandas as pd
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel

BASE = "/content/drive/MyDrive/Semantic_search2"
OUT = os.path.join(BASE, "outputs")
INDEX_DF = os.path.join(OUT, "index_df_canonical.csv")
SBERT_EMB = os.path.join(OUT, "nco_embeddings.npy")
MLP_PATH = os.path.join(OUT, "indicbert_mlp_mapper.pt")
INDICBERT_NAME = "ai4bharat/IndicBERTv2-MLM-only"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

assert os.path.exists(INDEX_DF) and os.path.exists(SBERT_EMB), "Run Part 2 (fine-tuned SBERT embeddings & FAISS) first."

print("Device:", DEVICE)
index_df = pd.read_csv(INDEX_DF, dtype=str).fillna("")
sbert_embs = np.load(SBERT_EMB)
texts = index_df["canonical_text"].fillna("").tolist()
print("Rows:", len(texts), "SBERT emb shape:", sbert_embs.shape)

# load IndicBERT
tokenizer = AutoTokenizer.from_pretrained(INDICBERT_NAME, use_fast=True)
indic_model = AutoModel.from_pretrained(INDICBERT_NAME).to(DEVICE)
indic_model.eval()

def mean_pool(out, mask):
    t = out.last_hidden_state
    mask_e = mask.unsqueeze(-1).expand(t.size()).float()
    s = torch.sum(t * mask_e, 1)
    d = mask_e.sum(1).clamp(min=1e-9)
    return s / d

# compute or reuse Indic embeddings
INDIC_EMB_PATH = os.path.join(OUT, "indic_embs_for_mlp.npy")
BATCH = 64  # lower to 16/32 if OOM
if os.path.exists(INDIC_EMB_PATH) and os.path.getmtime(INDIC_EMB_PATH) > os.path.getmtime(SBERT_EMB):
    print("Reusing existing Indic embeddings:", INDIC_EMB_PATH)
    indic_embs = np.load(INDIC_EMB_PATH)
else:
    print("Computing IndicBERT embeddings (this can take several minutes)...")
    embs = []
    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i+BATCH]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
        with torch.no_grad():
            out = indic_model(enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE))
            pooled = mean_pool(out, enc["attention_mask"].to(DEVICE)).cpu().numpy()
        embs.append(pooled)
    indic_embs = np.vstack(embs)
    np.save(INDIC_EMB_PATH, indic_embs)
    print("Saved Indic embeddings to", INDIC_EMB_PATH)

# Prepare dataset: map indic_embs -> normalized SBERT vectors
sbert_norm = sbert_embs / np.linalg.norm(sbert_embs, axis=1, keepdims=True).clip(min=1e-9)
X = torch.tensor(indic_embs, dtype=torch.float32)
Y = torch.tensor(sbert_norm, dtype=torch.float32)
dataset = TensorDataset(X, Y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# MLP definition
class MapperMLP(nn.Module):
    def __init__(self, inp_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, out_dim)
        )
    def forward(self, x):
        y = self.net(x)
        y = y / (torch.norm(y, dim=1, keepdim=True) + 1e-9)
        return y

inp_dim = X.shape[1]
out_dim = Y.shape[1]
mlp = MapperMLP(inp_dim, out_dim).to(DEVICE)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

EPOCHS = 8  # increase to 12 if you have time
print("Training MLP mapper:", inp_dim, "->", out_dim)
for epoch in range(EPOCHS):
    running_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = mlp(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        running_loss += loss.item() * xb.size(0)
    print(f"Epoch {epoch+1}/{EPOCHS} loss:", running_loss / len(dataset))

# save
torch.save({
    "state_dict": mlp.state_dict(),
    "inp_dim": inp_dim,
    "out_dim": out_dim,
    "indic_model": INDICBERT_NAME
}, MLP_PATH)
print("Saved MLP mapper ->", MLP_PATH)


Device: cuda
Rows: 3460 SBERT emb shape: (3460, 768)


tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Computing IndicBERT embeddings (this can take several minutes)...


  0%|          | 0/55 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Saved Indic embeddings to /content/drive/MyDrive/Semantic_search2/outputs/indic_embs_for_mlp.npy
Training MLP mapper: 768 -> 768
Epoch 1/8 loss: 0.0020637274116837565
Epoch 2/8 loss: 0.0017631252596391677
Epoch 3/8 loss: 0.0015283961788444332
Epoch 4/8 loss: 0.001391367389513664
Epoch 5/8 loss: 0.0013002905577442722
Epoch 6/8 loss: 0.0012353763696226169
Epoch 7/8 loss: 0.001181952273096308
Epoch 8/8 loss: 0.0011420317967233888
Saved MLP mapper -> /content/drive/MyDrive/Semantic_search2/outputs/indicbert_mlp_mapper.pt


# **LOAD TRAINED INDICBERT**

In [ ]:
# ===========================
# SINGLE ANCHOR TRAINING CELL
# - backs up existing MLP mapper
# - loads MLP mapper + IndicBERT + fine-tuned SBERT
# - trains on combined anchors (initial + booster)
# - saves updated MLP mapper (overwrites indicbert_mlp_mapper.pt but backup kept)
# ===========================

!pip install --quiet transformers sentence-transformers torch

import os, shutil, torch, numpy as np
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import torch.nn as nn

BASE = "/content/drive/MyDrive/Semantic_search2"
OUT = os.path.join(BASE, "outputs")
MLP_PATH = os.path.join(OUT, "indicbert_mlp_mapper.pt")
MLP_BAK = os.path.join(OUT, "indicbert_mlp_mapper.pt.bak")
SBERT_PATH = os.path.join(OUT, "sbert_nco_finetuned")

# Safety checks
assert os.path.exists(MLP_PATH), "MLP mapper not found. Train MLP first."
assert os.path.exists(SBERT_PATH), "Fine-tuned SBERT not found. Run fine-tune first."

# Backup existing mapper if not already backed up
if os.path.exists(MLP_PATH) and not os.path.exists(MLP_BAK):
    shutil.copy(MLP_PATH, MLP_BAK)
    print("Backed up existing mapper ->", MLP_BAK)
else:
    print("Backup already exists or mapper missing (backup skipped).")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---- load existing MLP mapper ----
ckpt = torch.load(MLP_PATH, map_location=DEVICE)
inp_dim = ckpt.get("inp_dim", None)
out_dim = ckpt.get("out_dim", None)
indic_model_name = ckpt.get("indic_model", "ai4bharat/IndicBERTv2-MLM-only")

# define same architecture
class MapperMLP(nn.Module):
    def __init__(self, inp, out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp, 1024),
            nn.ReLU(),
            nn.Linear(1024, out)
        )
    def forward(self, x):
        y = self.net(x)
        # normalization helps when target SBERT vectors are normalized
        return y / (torch.norm(y, dim=1, keepdim=True) + 1e-9)

# instantiate and load weights
if inp_dim is None or out_dim is None:
    # fallback assumption
    inp_dim = 768; out_dim = 768
mlp = MapperMLP(inp_dim, out_dim).to(DEVICE)
mlp.load_state_dict(ckpt["state_dict"])
mlp.train()
print("Loaded MLP mapper. inp_dim:", inp_dim, "out_dim:", out_dim)

# ---- load IndicBERT tokenizer+model + SBERT ----
tokenizer = AutoTokenizer.from_pretrained(indic_model_name, use_fast=True)
indic_model = AutoModel.from_pretrained(indic_model_name).to(DEVICE)
indic_model.eval()

sbert = SentenceTransformer(SBERT_PATH)
sbert.eval()

# mean-pooling helper for IndicBERT outputs (robust)
def mean_pool(output, mask):
    token_embeddings = output.last_hidden_state
    mask_exp = mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_emb = (token_embeddings * mask_exp).sum(1)
    denom = mask_exp.sum(1).clamp(min=1e-9)
    return sum_emb / denom

# ---- COMBINED ANCHORS (initial + strong booster) ----
anchors = [
    # initial anchors (useful)
    ("सिलाई ऑपरेटर", "Sewing Machine Operator"),
    ("सिलाई मशीन ऑपरेटर", "Sewing Machine Operator"),
    ("दरजी", "Tailor"),
    ("सिलाई करने वाला", "Sewing Machine Operator"),
    ("सिले कपड़े बनाने वाला", "Tailor"),
    ("शिवण मशीन", "Sewing Machine Operator"),
    ("ट्रक ड्राइवर", "Driver Truck"),
    ("लॉरी ड्राइवर", "Heavy Truck and Lorry Drivers"),
    ("बस ड्राइवर", "Bus Driver"),
    ("सॉफ्टवेयर इंजीनियर", "Software Engineer"),
    ("कंप्यूटर प्रोग्रामर", "Programming Assistant"),
    ("एप डेवलपर", "Software Developer"),
    ("सुरक्षा गार्ड", "Security Guard"),
    ("सिक्योरिटी ऑफिसर", "Security Officer"),
    ("चौकीदार", "Security Guard"),
    ("शिक्षक", "Teacher"),
    ("स्कूल शिक्षक", "Primary School Teacher"),
    # booster anchors (strong variants)
    ("सिलाई मशीन चलाने वाला", "Sewing Machine Operator"),
    ("सिलाई मशीन वर्कर", "Sewing Machine Operator"),
    ("सिलाई मशीन कर्मचारी", "Sewing Machine Operator"),
    ("मशीन से सिलाई करने वाला", "Sewing Machine Operator"),
    ("कपड़े सिलने वाला", "Tailor"),
    ("टेलर", "Tailor"),
    ("कपड़े की सिलाई करने वाला", "Tailor"),
    ("दरजी", "Tailor"),  # duplicate ok, reinforces anchor
    ("लॉरी ड्राइवर", "Heavy Truck and Lorry Drivers"),
    ("ट्रक ड्राइवर", "Driver Truck"),
]

print("Total anchors:", len(anchors))

# ---- Build training tensors for anchors ----
X_list, Y_list = [], []
for hin, eng in anchors:
    enc = tokenizer([hin], padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        out = indic_model(enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE))
        pooled = mean_pool(out, enc["attention_mask"].to(DEVICE)).cpu().numpy()
    # SBERT target normalized vector
    tgt = sbert.encode([eng], normalize_embeddings=True)
    X_list.append(pooled[0])
    Y_list.append(tgt[0])

X = torch.tensor(np.array(X_list), dtype=torch.float32).to(DEVICE)
Y = torch.tensor(np.array(Y_list), dtype=torch.float32).to(DEVICE)

print("Anchor tensors prepared:", X.shape, Y.shape)

# ---- Small training loop (gentle LR; few epochs to avoid damaging generalization) ----
opt = torch.optim.Adam(mlp.parameters(), lr=5e-6)   # small LR
loss_fn = nn.MSELoss()
EPOCHS = 18
print("Starting anchor fine-tune for", EPOCHS, "epochs...")

for epoch in range(EPOCHS):
    opt.zero_grad()
    pred = mlp(X)
    loss = loss_fn(pred, Y)
    loss.backward()
    opt.step()
    if (epoch+1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}  loss={loss.item():.6e}")

# ---- Save updated mapper ----
torch.save({
    "state_dict": mlp.state_dict(),
    "inp_dim": inp_dim,
    "out_dim": out_dim,
    "indic_model": indic_model_name
}, MLP_PATH)

print("Anchor training complete. Updated mapper saved to:", MLP_PATH)
print("Backup (original) is at:", MLP_BAK)


Backed up existing mapper -> /content/drive/MyDrive/Semantic_search2/outputs/indicbert_mlp_mapper.pt.bak
Device: cuda
Loaded MLP mapper. inp_dim: 768 out_dim: 768
Total anchors: 27
Anchor tensors prepared: torch.Size([27, 768]) torch.Size([27, 768])
Starting anchor fine-tune for 18 epochs...
Epoch 1/18  loss=1.819390e-03
Epoch 3/18  loss=1.797955e-03
Epoch 6/18  loss=1.766210e-03
Epoch 9/18  loss=1.734997e-03
Epoch 12/18  loss=1.704332e-03
Epoch 15/18  loss=1.674252e-03
Epoch 18/18  loss=1.644756e-03
Anchor training complete. Updated mapper saved to: /content/drive/MyDrive/Semantic_search2/outputs/indicbert_mlp_mapper.pt
Backup (original) is at: /content/drive/MyDrive/Semantic_search2/outputs/indicbert_mlp_mapper.pt.bak


In [ ]:
# ============================================================
# NEW SMOKE TEST (AFTER ANCHOR TRAINING)
# ============================================================

import os
import torch
import faiss
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel

BASE = "/content/drive/MyDrive/Semantic_search2"
OUT = os.path.join(BASE, "outputs")

MLP_PATH = os.path.join(OUT, "indicbert_mlp_mapper.pt")
FAISS_IDX = os.path.join(OUT, "nco_faiss.index")
INDEX_DF = os.path.join(OUT, "index_df_canonical.csv")

assert os.path.exists(MLP_PATH), "MLP mapper not found — run anchor training first!"
assert os.path.exists(FAISS_IDX), "FAISS index missing — run Part 2 first."

# --------------------------------------------------------------------
# Load index + FAISS
# --------------------------------------------------------------------
index_df = pd.read_csv(INDEX_DF, dtype=str).fillna("")
fa = faiss.read_index(FAISS_IDX)

# --------------------------------------------------------------------
# Load MLP mapper (updated after anchor training)
# --------------------------------------------------------------------
ckpt = torch.load(MLP_PATH, map_location="cuda" if torch.cuda.is_available() else "cpu")
inp_dim = ckpt["inp_dim"]
out_dim = ckpt["out_dim"]
INDIC_NAME = ckpt["indic_model"]

class MapperMLP(torch.nn.Module):
    def __init__(self, inp, out):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(inp, 1024),
            torch.nn.ReLU(),
            torch.nn.Linear(1024, out),
        )
    def forward(self, x):
        y = self.net(x)
        return y / (torch.norm(y, dim=1, keepdim=True) + 1e-9)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

mlp = MapperMLP(inp_dim, out_dim).to(DEVICE)
mlp.load_state_dict(ckpt["state_dict"])
mlp.eval()

# --------------------------------------------------------------------
# Load IndicBERT
# --------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(INDIC_NAME, use_fast=True)
indic_model = AutoModel.from_pretrained(INDIC_NAME).to(DEVICE)
indic_model.eval()

def mean_pool(output, mask):
    tok = output.last_hidden_state
    mask_e = mask.unsqueeze(-1).expand(tok.size()).float()
    return (tok * mask_e).sum(1) / mask_e.sum(1).clamp(min=1e-9)

# --------------------------------------------------------------------
# Encode Indic query → mapped SBERT vector
# --------------------------------------------------------------------
def indic_to_sbert_vec(q):
    enc = tokenizer([q], padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        out = indic_model(enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE))
        pooled = mean_pool(out, enc["attention_mask"].to(DEVICE)).cpu()
    with torch.no_grad():
        mapped = mlp(pooled.to(DEVICE)).cpu().numpy()
    return mapped

def search(q, k=5):
    vec = indic_to_sbert_vec(q)
    D, I = fa.search(vec.reshape(1, -1), k)
    rows = []
    for score, idx in zip(D[0], I[0]):
        row = index_df.iloc[idx]
        rows.append({
            "score": round(float(score), 4),
            "NCO_Code": row["NCO_Code"],
            "Title": row["Title"],
        })
    return pd.DataFrame(rows)

# --------------------------------------------------------------------
# TEST QUERIES (change freely)
# --------------------------------------------------------------------
queries = [
    "सिलाई ऑपरेटर",
    "ट्रक ड्राइवर",
    "सुरक्षा गार्ड",
    "सॉफ्टवेयर इंजीनियर",
    "दरजी",
    "कंप्यूटर प्रोग्रामर",
]

for q in queries:
    print("\n🔍 Query:", q)
    display(search(q))



🔍 Query: सिलाई ऑपरेटर


,score,NCO_Code,Title
0,0.5635,815209,"Knitter, Machine/Knitting Machine\nOperator"
1,0.5323,81223,Reeling Machine Operator
2,0.5259,,Machine Operators
3,0.5150,81530101,"Sewing Machine Operator, General"
4,0.5112,413206,Coding Machine Operator



🔍 Query: ट्रक ड्राइवर


,score,NCO_Code,Title
0,0.3861,,Machine Operators
1,0.3839,833201,Driver Truck
2,0.3662,83220501,Light Motor Vehicle Driver
3,0.3608,832201,"Driver, Car"
4,0.3516,831101,"Loco Driver, Mines"



🔍 Query: सुरक्षा गार्ड


...

,score,NCO_Code,Title
0,0.3078,,"Machine Operators, Other"
1,0.3035,35120801,Engineer – Technical Support
2,0.3012,,Machine Operators
3,0.2958,31141301,Network Management Engineer
4,0.2949,,Operator



🔍 Query: दरजी


,score,NCO_Code,Title
0,0.4332,,Operators
1,0.4179,,Operator
2,0.4179,,Operator
3,0.3749,,Operators Not Elsewhere Classified
4,0.3515,,"Machine Operators, Other"



🔍 Query: कंप्यूटर प्रोग्रामर


,score,NCO_Code,Title
0,0.4125,413206,Coding Machine Operator
1,0.4065,,Machine Operators
2,0.4008,351403,Programming Assistant/Junior Software\nEngineer
3,0.3972,251901,System Programmer
4,0.3904,251208,"Programmer, Engineering and Scientific/System\..."


**GRADIO**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# FINAL LOAD CELL

!pip install -q sentence-transformers transformers faiss-cpu gradio

import os, re, json, torch, faiss, pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

OUT_DIR = "/content/drive/MyDrive/Semantic_search2/outputs"
FINETUNED_SBERT = os.path.join(OUT_DIR, "sbert_nco_finetuned")
FAISS_INDEX_PATH = os.path.join(OUT_DIR, "nco_faiss.index")
INDEX_DF_PATH = os.path.join(OUT_DIR, "index_df_canonical.csv")
MLP_PATH = os.path.join(OUT_DIR, "indicbert_mlp_mapper.pt")


# Load SBERT (prefer fine-tuned)

if os.path.exists(FINETUNED_SBERT):
    print("Loading FINE-TUNED SBERT:", FINETUNED_SBERT)
    sbert = SentenceTransformer(FINETUNED_SBERT)
else:
    print("FINE-TUNED SBERT NOT FOUND! Loading base SBERT.")
    sbert = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
sbert.max_seq_length = 256


# Load FAISS index

assert os.path.exists(FAISS_INDEX_PATH), "FAISS index missing!"
faiss_index = faiss.read_index(FAISS_INDEX_PATH)
print("FAISS loaded. Vectors:", faiss_index.ntotal)


# Load index_df

assert os.path.exists(INDEX_DF_PATH), "index_df missing!"
index_df = pd.read_csv(INDEX_DF_PATH, dtype=str).fillna("")
print("index_df loaded. Rows:", len(index_df))


# Load IndicBERT + MLP mapper (if exists)

USE_INDIC = os.path.exists(MLP_PATH)

if USE_INDIC:
    print("Loading IndicBERT + MLP mapper...")
    ckpt = torch.load(MLP_PATH, map_location="cpu")

    inp_dim = ckpt["inp_dim"]
    out_dim = ckpt["out_dim"]
    indic_model_name = ckpt.get("indic_model", "ai4bharat/IndicBERTv2-MLM-only")

    indic_tokenizer = AutoTokenizer.from_pretrained(indic_model_name)
    indic_model = AutoModel.from_pretrained(indic_model_name)
    indic_model.eval()

    class MapperMLP(torch.nn.Module):
        def __init__(self, inp, out):
            super().__init__()
            self.net = torch.nn.Sequential(
                torch.nn.Linear(inp, 1024),
                torch.nn.ReLU(),
                torch.nn.Linear(1024, out)
            )
        def forward(self, x):
            y = self.net(x)
            return y / (torch.norm(y, dim=1, keepdim=True) + 1e-9)

    mlp = MapperMLP(inp_dim, out_dim)
    mlp.load_state_dict(ckpt["state_dict"])
    mlp.eval()

    print("IndicBERT + MLP loaded.")
else:
    indic_tokenizer = None
    indic_model = None
    mlp = None
    print("No MLP found → Indic pipeline disabled.")


# MULTI-SCRIPT DETECTOR

indic_scripts_re = re.compile(
    r'['
    r'\u0900-\u097F'  # Devanagari (Hindi/Marathi)
    r'\u0980-\u09FF'  # Bengali/Assamese
    r'\u0A00-\u0A7F'  # Gurmukhi (Punjabi)
    r'\u0A80-\u0AFF'  # Gujarati
    r'\u0B00-\u0B7F'  # Oriya
    r'\u0B80-\u0BFF'  # Tamil
    r'\u0C00-\u0C7F'  # Telugu
    r'\u0C80-\u0CFF'  # Kannada
    r'\u0D00-\u0D7F'  # Malayalam
    r'\u0600-\u06FF'  # Urdu
    r']'
)

print("Multi-script detector added (ALL major Indic languages).")


# Define helper functions

def faiss_search(vec, k=5):
    D,I = faiss_index.search(np.array([vec]).astype("float32"), k)
    out = []
    for score, idx in zip(D[0], I[0]):
        rec = index_df.iloc[idx]
        out.append({
            "score": float(score),
            "NCO_Code": rec["NCO_Code"],
            "Title": rec["Title"],
            "Description": (rec.get("Description","") or "")[:300]
        })
    return out

def encode_indic(q):
    enc = indic_tokenizer([q], truncation=True, padding=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        out = indic_model(**enc)
        mask = enc["attention_mask"]
        hidden = out.last_hidden_state
        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1).unsqueeze(-1)
        mapped = mlp(pooled).cpu().numpy()[0]
        return mapped

def encode_sbert(q):
    return sbert.encode([q], normalize_embeddings=True)[0]

print("\n🔥 ALL MODELS LOADED SUCCESSFULLY")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 63.8 MB/s eta 0:00:00
Loading FINE-TUNED SBERT: /content/drive/MyDrive/Semantic_search2/outputs/sbert_nco_finetuned
FAISS loaded. Vectors: 3460
index_df loaded. Rows: 3460
Loading IndicBERT + MLP mapper...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

IndicBERT + MLP loaded.
Multi-script detector added (ALL major Indic languages).

🔥 ALL MODELS LOADED SUCCESSFULLY


In [5]:
# Gradio UI Cell

import gradio as gr
import pandas as pd

def search_occupations_auto(query, top_k=5):
    q = str(query or "").strip()
    if not q:
        return "Please enter a query.", None

    # detect Indic language
    is_indic = bool(indic_scripts_re.search(q))

    # If Indic + MLP available → use IndicBERT pipeline
    if is_indic and USE_INDIC:
        vec = encode_indic(q)
        rows = faiss_search(vec, top_k)
        return "IndicBERT + MLP pipeline", pd.DataFrame(rows)

    # Otherwise → English SBERT (fine-tuned)
    vec = encode_sbert(q)
    rows = faiss_search(vec, top_k)
    return "SBERT pipeline", pd.DataFrame(rows)

demo = gr.Interface(
    fn=search_occupations_auto,
    inputs=[
        gr.Textbox(label="Enter job title or description"),
        gr.Slider(1, 10, value=5, label="Number of results")
    ],
    outputs=[
        gr.Textbox(label="Summary"),
        gr.Dataframe(label="Top Matches")
    ],
    title="NCO Semantic Search (Full Indic Support)",
    description="Auto-detects Hindi/Bengali/Tamil/Telugu/Urdu/etc. Uses IndicBERT or fine-tuned SBERT."
)

demo.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://94acd3e52849a1e8da.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://94acd3e52849a1e8da.gradio.live


**VALIDATION**

In [6]:
# run this cell AFTER loading your index_df (index_df_canonical.csv)
import os, random, pandas as pd, re, json
OUT = "/content/drive/MyDrive/Semantic_search2/outputs"
INDEX_DF = os.path.join(OUT, "index_df_canonical.csv")
EVAL_CSV = os.path.join(OUT, "eval_queries.csv")

df = pd.read_csv(INDEX_DF, dtype=str).fillna("")

# A. Self-retrieval: use Title as query
rows = []
for _, r in df.iterrows():
    title = (r.get("Title") or "").strip()
    nco = (r.get("NCO_Code") or "").strip()
    if title and nco:
        rows.append({"query": title, "correct_code": nco, "source": "self_title"})

# B. Synthetic variations: truncate, lowercase, add short prefix/suffix
def variations(title):
    title = title.strip()
    v = set()
    v.add(title)
    # lowercase/truncate
    tokens = title.split()
    if len(tokens) > 1:
        v.add(" ".join(tokens[:2]))   # first 2 words
        v.add(" ".join(tokens[:3]))   # first 3 words
    v.add(title.lower())
    # short paraphrase-like suffixes
    v.add(title + " job")
    v.add("skilled " + title)
    return list(v)

# add a few synthetic variants for up to N rows (to limit explosion)
N = 1000   # increase if you want more queries
for i, r in df.head(N).iterrows():
    title = (r.get("Title") or "").strip()
    nco = (r.get("NCO_Code") or "").strip()
    for v in variations(title)[:4]:
        rows.append({"query": v, "correct_code": nco, "source": "synthetic"})

# dedupe and save
seen = set()
uniq = []
for r in rows:
    key = (r["query"].strip().lower(), r["correct_code"])
    if key in seen:
        continue
    seen.add(key)
    uniq.append(r)

eval_df = pd.DataFrame(uniq)
eval_df.to_csv(EVAL_CSV, index=False, encoding="utf8")
print("Saved eval CSV to:", EVAL_CSV)
print("Rows:", len(eval_df))
eval_df.head()

Saved eval CSV to: /content/drive/MyDrive/Semantic_search2/outputs/eval_queries.csv
Rows: 5788


,query,correct_code,source
0,"Elected Official, Union Government",111101,self_title
1,"Elected Official, State Government",111102,self_title
2,"Elected Official, Local Bodies",111103,self_title
3,"Legislators, Other",111199,self_title
4,"Administrative Official, Union Government",111201,self_title


In [13]:
# Full evaluation cell
!pip install -q sentence-transformers transformers faiss-cpu tqdm numpy pandas

import os, json, numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

BASE = "/content/drive/MyDrive/Semantic_search2"
OUT = os.path.join(BASE, "outputs")

# Files (adjust if needed)
INDEX_DF = os.path.join(OUT, "index_df_canonical.csv")
EVAL_CSV = os.path.join(OUT, "eval_queries.csv")
FAISS_PATH = os.path.join(OUT, "nco_faiss.index")
SBERT_EMB = os.path.join(OUT, "nco_embeddings.npy")
MAPPER_PATH = os.path.join(OUT, "indicbert_mlp_mapper.pt")
FINETUNED_SBERT_DIR = os.path.join(OUT, "sbert_nco_finetuned")

assert os.path.exists(INDEX_DF), "Missing index_df"
assert os.path.exists(EVAL_CSV), "Create eval CSV first"
assert os.path.exists(FAISS_PATH), "Missing FAISS index"

index_df = pd.read_csv(INDEX_DF, dtype=str).fillna("")
eval_df = pd.read_csv(EVAL_CSV, dtype=str).fillna("")
# a list mapping faiss index position -> NCO_Code
# If you stored codes separately, load them; else use index_df order
code_list = index_df['NCO_Code'].astype(str).tolist()

# load faiss
import faiss
fa = faiss.read_index(FAISS_PATH)
print("FAISS vectors:", fa.ntotal, "index_df rows:", len(index_df))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ---------------------------
# Helper encoders
# ---------------------------

# 1) SBERT base
sbert_base = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
sbert_base.max_seq_length = 256
sbert_base = sbert_base.to(device)

# 2) SBERT fine-tuned
if os.path.exists(FINETUNED_SBERT_DIR):
    sbert_ft = SentenceTransformer(FINETUNED_SBERT_DIR)
    sbert_ft.max_seq_length = 256
    sbert_ft = sbert_ft.to(device)
else:
    print("Fine-tuned SBERT not found, using base for ft placeholder.")
    sbert_ft = sbert_base

# 3) IndicBERT raw (mean-pool)
INDIC_NAME = "ai4bharat/IndicBERTv2-MLM-only"  # same used previously
tokenizer_indic = AutoTokenizer.from_pretrained(INDIC_NAME, use_fast=True)
indic_model = AutoModel.from_pretrained(INDIC_NAME).to(device)
indic_model.eval()

def mean_pool_hidden(output, mask):
    t = output.last_hidden_state
    mask_e = mask.unsqueeze(-1).expand(t.size()).float()
    s = torch.sum(t * mask_e, 1)
    d = mask_e.sum(1).clamp(min=1e-9)
    return s / d

# 4) IndicBERT -> SBERT mapped: load MLP mapper
mapper = None
if os.path.exists(MAPPER_PATH):
    ckpt = torch.load(MAPPER_PATH, map_location=device)
    inp_dim = ckpt.get("inp_dim")
    out_dim = ckpt.get("out_dim")
    # define same architecture
    import torch.nn as nn
    class MapperMLP(nn.Module):
        def __init__(self, inp, out):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(inp, 1024),
                nn.ReLU(),
                nn.Linear(1024, out)
            )
        def forward(self, x):
            y = self.net(x)
            return y / (torch.norm(y, dim=1, keepdim=True) + 1e-9)
    mapper = MapperMLP(ckpt.get("inp_dim", 768), ckpt.get("out_dim", 768)).to(device)
    mapper.load_state_dict(ckpt["state_dict"])
    mapper.eval()
    print("Loaded mapper:", MAPPER_PATH)
else:
    print("Mapper not found at:", MAPPER_PATH)

# ---------------------------
# Encoding functions (return numpy float32 normalized vectors)
# ---------------------------
import numpy as np
import torch.nn.functional as F

def encode_sbert_base(batch_queries):
    emb = sbert_base.encode(
        batch_queries,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return emb.astype("float32")

def encode_sbert_ft(batch_queries):
    emb = sbert_ft.encode(
        batch_queries,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return emb.astype("float32")

def encode_indic_raw(batch_queries):
    batch = tokenizer_indic(batch_queries, padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        out = indic_model(batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
        pooled = mean_pool_hidden(out, batch["attention_mask"].to(device)).cpu().numpy()
    # normalize
    norms = np.linalg.norm(pooled, axis=1, keepdims=True).clip(min=1e-9)
    return (pooled / norms).astype("float32")

def encode_indic_mapped(batch_queries):
    # produce indic raw then map via mapper
    batch = tokenizer_indic(batch_queries, padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        out = indic_model(batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
        pooled = mean_pool_hidden(out, batch["attention_mask"].to(device)).to(device)
        mapped = mapper(pooled).cpu().numpy()
    norms = np.linalg.norm(mapped, axis=1, keepdims=True).clip(min=1e-9)
    return (mapped / norms).astype("float32")

# ---------------------------
# Batched evaluation loop
# ---------------------------
from collections import defaultdict
def evaluate(encode_fn, queries, gt_codes, topk=(1,3,5,10), batch_size=32):
    assert len(queries) == len(gt_codes)
    n = len(queries)
    counts = defaultdict(int)
    mrr = 0.0
    # we will also collect per-query results optionally
    per_results = []
    for i in tqdm(range(0, n, batch_size)):
        batch_q = queries[i:i+batch_size]
        batch_embs = encode_fn(batch_q)   # shape (B, dim)
        # for each embedding, search faiss
        D, I = fa.search(batch_embs.astype("float32"), max(topk))
        for j in range(len(batch_q)):
            retrieved_codes = [code_list[idx] for idx in I[j].tolist()]
            correct = str(gt_codes[i+j]).strip()
            # compute Top-K hits
            for k in topk:
                if correct in retrieved_codes[:k]:
                    counts[f"recall@{k}"] += 1
            # MRR
            if correct in retrieved_codes:
                rank = retrieved_codes.index(correct) + 1
                mrr += 1.0 / rank
                found_rank = rank
            else:
                found_rank = None
            per_results.append({
                "query": batch_q[j],
                "correct": correct,
                "retrieved": json.dumps(retrieved_codes[:max(topk)]),
                "found_rank": found_rank
            })
    # finalize metrics
    metrics = {}
    for k in topk:
        metrics[f"recall@{k}"] = counts[f"recall@{k}"] / n
    metrics["MRR"] = mrr / n
    return metrics, pd.DataFrame(per_results)

# ---------------------------
# Run evaluation for all 4 models
# ---------------------------
eval_queries = eval_df['query'].astype(str).tolist()
eval_codes = eval_df['correct_code'].astype(str).tolist()
print("Eval size:", len(eval_queries))

results = {}
details = {}

# 1 SBERT base
print("Evaluating SBERT base...")
res, df_res = evaluate(encode_sbert_base, eval_queries, eval_codes, topk=(1,3,5,10), batch_size=64)
results["SBERT_base"] = res; details["SBERT_base"] = df_res
print(res)

# 2 SBERT fine-tuned
print("Evaluating SBERT fine-tuned...")
res, df_res = evaluate(encode_sbert_ft, eval_queries, eval_codes, topk=(1,3,5,10), batch_size=64)
results["SBERT_ft"] = res; details["SBERT_ft"] = df_res
print(res)

# 3 IndicBERT raw
print("Evaluating IndicBERT raw...")
res, df_res = evaluate(encode_indic_raw, eval_queries, eval_codes, topk=(1,3,5,10), batch_size=32)
results["Indic_raw"] = res; details["Indic_raw"] = df_res
print(res)

# 4 IndicBERT -> mapped
if mapper is not None:
    print("Evaluating IndicBERT -> mapped...")
    res, df_res = evaluate(encode_indic_mapped, eval_queries, eval_codes, topk=(1,3,5,10), batch_size=32)
    results["Indic_mapped"] = res; details["Indic_mapped"] = df_res
    print(res)
else:
    print("Skipping mapped evaluation (mapper missing).")

# Save summary
res_df = pd.DataFrame([{ "model": k, **v } for k,v in results.items()])
res_df.to_csv(os.path.join(OUT, "eval_summary.csv"), index=False)
# Save per-query details for each model
for k,v in details.items():
    v.to_csv(os.path.join(OUT, f"eval_details_{k}.csv"), index=False)

print("Evaluation complete. Saved summary to:", os.path.join(OUT, "eval_summary.csv"))
print(res_df)


FAISS vectors: 3460 index_df rows: 3460
Device: cuda
Loaded mapper: /content/drive/MyDrive/Semantic_search2/outputs/indicbert_mlp_mapper.pt
Eval size: 5788
Evaluating SBERT base...


  0%|          | 0/91 [00:00<?, ?it/s]

{'recall@1': 0.8564270905321355, 'recall@3': 0.9457498272287491, 'recall@5': 0.962681409813407, 'recall@10': 0.9787491361437457, 'MRR': 0.903555522646745}
Evaluating SBERT fine-tuned...


  0%|          | 0/91 [00:00<?, ?it/s]

{'recall@1': 0.9080856945404284, 'recall@3': 0.9621630960608155, 'recall@5': 0.9732204561161023, 'recall@10': 0.9832411886662059, 'MRR': 0.9365319325149998}
Evaluating IndicBERT raw...


  0%|          | 0/181 [00:00<?, ?it/s]

{'recall@1': 0.0003455425017277125, 'recall@3': 0.000691085003455425, 'recall@5': 0.0008638562543192813, 'recall@10': 0.002073255010366275, 'MRR': 0.0006989694057765929}
Evaluating IndicBERT -> mapped...


  0%|          | 0/181 [00:00<?, ?it/s]

{'recall@1': 0.085348997926745, 'recall@3': 0.17760884588804424, 'recall@5': 0.2322045611610228, 'recall@10': 0.32394609536973046, 'MRR': 0.14945481071949662}
Evaluation complete. Saved summary to: /content/drive/MyDrive/Semantic_search2/outputs/eval_summary.csv
          model  recall@1  recall@3  recall@5  recall@10       MRR
0    SBERT_base  0.856427  0.945750  0.962681   0.978749  0.903556
1      SBERT_ft  0.908086  0.962163  0.973220   0.983241  0.936532
2     Indic_raw  0.000346  0.000691  0.000864   0.002073  0.000699
3  Indic_mapped  0.085349  0.177609  0.232205   0.323946  0.149455
